# Reacting-flow fundamentals

The building blocks behind the combustor examples:

1. **`nefes.thermo` HP equilibrium** -- the adiabatic flame temperature vs.
   equivalence ratio, straight from the NASA `thermo.inp` data (no network);
2. the perfect-gas **heat-release flame** -- a prescribed power `Qdot` raising the
   total temperature;
3. the reacting **equilibrium flame** -- frozen unburnt reactants in, equilibrium
   products out, the flame temperature emerging from the solve.


In [ ]:
import os, sys
_root = os.getcwd()
while not os.path.isdir(os.path.join(_root, "nefes")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)

import numpy as np
import plotly.graph_objects as go

from nefes.thermo import ThermoInp, Thermo
from nefes.chem.composition import resolve_composition, enthalpy_mass, equivalence_ratio_mixture
from nefes.elements import catalog as cat
from nefes.shell import build_problem
from nefes.solver import solve
from nefes.solver.report import states_table
from nefes.thermo.api import EQ_FROZEN, EQ_KERNEL
from nefes.thermo.configure import equilibrium, perfect_gas
from nefes.assembly.recover import ES_MDOT, ES_P, ES_HT, ES_RHO, ES_U, ES_T, ES_M

THERMO_INP = os.path.join(_root, "nefes", "thermo", "data", "thermo.inp")
R_AIR, GAMMA = 287.0, 1.4
CP = GAMMA * R_AIR / (GAMMA - 1.0)

from nefes.plotting import use_nefes_theme, COLORWAY
use_nefes_theme()

## 1. Adiabatic flame temperature vs. equivalence ratio (thermochemistry only)
Standalone chemical equilibrium: build a small H2/air library, then solve HP
equilibrium of the reactant elements at the reactant enthalpy.

In [ ]:
lib = ThermoInp(THERMO_INP).library(["H2", "O2", "N2", "H2O", "OH", "H", "O", "HO2", "NO"])
gas = Thermo(lib)

AIR = {"O2": 0.21, "N2": 0.79}  # dry air (mole)

def h2_air(phi):
    """H2/air mixture at equivalence ratio phi (mole), via the Nefes mixture helper.

    `equivalence_ratio_mixture` fixes the stoichiometric H2/air ratio from the
    elemental oxygen balance -- no hand-tuned 0.5 O2 / 3.76 N2 bookkeeping.
    """
    return equivalence_ratio_mixture(lib, {"H2": 1.0}, AIR, phi)

phis = np.linspace(0.4, 1.6, 13)
Tad = []
for phi in phis:
    Y, Z = resolve_composition(lib, h2_air(phi), basis="mole")
    h = enthalpy_mass(lib, Y, 300.0)
    eq = gas.equilibrate_HP(Z, h, 101325.0, T_guess=2000.0)
    Tad.append(eq.T)

fig = go.Figure()
fig.add_scatter(x=phis, y=Tad, mode="lines+markers", line_color=COLORWAY[4])
fig.update_layout(
    title="H₂/air adiabatic flame temperature (HP equilibrium)",
    xaxis_title="equivalence ratio  φ", yaxis_title="adiabatic flame T  [K]",
    showlegend=False, height=440,
)
fig.show()
print(f"peak Tad = {max(Tad):.1f} K near phi = {phis[int(np.argmax(Tad))]:.2f}")

## 2. Perfect-gas heat-release flame
A 2-port flame that adds a prescribed power `Qdot` [W]; the total temperature
rises by `Qdot / (mdot * cp)`, mass and the constant-area momentum flux are
conserved (a Rayleigh static-pressure drop appears).

In [ ]:
mdot, Tt, Qdot, A = 10.0, 300.0, 2.0e6, 0.05
cfg = perfect_gas(R_AIR, GAMMA)
els = [cat.mass_flow_inlet(mdot, Tt), cat.heat_release_flame(Qdot), cat.pressure_outlet(1.0e5)]
prob = build_problem(cfg, els, [(0, 1, A), (1, 2, A)], mdot_ref=mdot, p_ref=1.0e5, h_ref=CP * Tt)
res = solve(prob); est = states_table(prob, res.x)
print("converged:", res.converged)
print(f"  Tt in/out : {est[ES_HT,0]/CP:.1f} -> {est[ES_HT,1]/CP:.1f} K  (predicted +{Qdot/(mdot*CP):.1f} K)")
print(f"  static T  : {est[ES_T,0]:.1f} -> {est[ES_T,1]:.1f} K")
print(f"  static p  : {est[ES_P,0]/1e3:.2f} -> {est[ES_P,1]/1e3:.2f} kPa (Rayleigh drop)")
imp = [est[ES_RHO,e]*est[ES_U,e]**2 + est[ES_P,e] for e in (0,1)]
print(f"  momentum flux rho*u^2 + p conserved: {imp[0]:.1f} vs {imp[1]:.1f} Pa")

## 3. Reacting equilibrium flame (single premixed stream)
Frozen H2/air approach -> equilibrium products.  'Ignition' is the per-edge
closure switch: the approach edge is `EQ_FROZEN`, the product edge `EQ_KERNEL`.
The premixed H2/air is a single **feed stream** (one transported mixture fraction,
discovered from the inlet composition at build time), and `solve(prob)` seeds
itself by propagating the feed through the network -- no hand-built guess.

In [ ]:
phi = 1.0
Y, Z = resolve_composition(lib, h2_air(phi), basis="mole")
h_react = enthalpy_mass(lib, Y, 300.0)
mdot, p, A = 1.0, 101325.0, 0.05
cfg = equilibrium(lib)
els = [
    cat.mass_flow_inlet(mdot, 300.0, composition=h2_air(phi), name="fuel-air"),
    cat.equilibrium_flame(name="flame"),
    cat.pressure_outlet(p, Tt_backflow=300.0, composition=h2_air(phi), name="out"),
]
prob = build_problem(cfg, els, [(0, 1, A), (1, 2, A)], mdot_ref=mdot, p_ref=p,
                         h_ref=max(abs(h_react), 1e4), edge_models=[EQ_FROZEN, EQ_KERNEL])
res = solve(prob); est = states_table(prob, res.x)
ref = gas.equilibrate_HP(Z, h_react, est[ES_P, 1], T_guess=2000.0)
print("converged:", res.converged)
print(f"  unburnt T : {est[ES_T,0]:.1f} K     burnt T : {est[ES_T,1]:.1f} K")
print(f"  standalone HP-equilibrium reference : {ref.T:.1f} K")
print(f"  density dilatation rho_u/rho_b = {est[ES_RHO,0]/est[ES_RHO,1]:.2f}")

In [ ]:
from nefes.shell import Network, Solution

network = Network(cfg, p_ref=p, mdot_ref=mdot, h_ref=max(abs(h_react), 1e4),
                  nodes=els, edges=[(0, 1, A), (1, 2, A)], edge_models=[EQ_FROZEN, EQ_KERNEL])
sol = Solution(network, prob, res)

## Equivalence-ratio sweep through the network
The network flame temperature tracks the standalone adiabatic flame temperature.

In [ ]:
phis2 = np.linspace(0.5, 1.4, 10)
Tnet, Tref = [], []
for phi in phis2:
    Y, Z = resolve_composition(lib, h2_air(phi), basis="mole")
    h_react = enthalpy_mass(lib, Y, 300.0)
    els = [cat.mass_flow_inlet(1.0, 300.0, composition=h2_air(phi)),
           cat.equilibrium_flame(),
           cat.pressure_outlet(101325.0, Tt_backflow=300.0, composition=h2_air(phi))]
    prob = build_problem(cfg, els, [(0,1,0.05),(1,2,0.05)], mdot_ref=1.0, p_ref=101325.0,
                             h_ref=max(abs(h_react),1e4), edge_models=[EQ_FROZEN, EQ_KERNEL])
    r = solve(prob)
    e = states_table(prob, r.x)
    Tnet.append(e[ES_T, 1])
    Tref.append(gas.equilibrate_HP(Z, h_react, e[ES_P, 1], T_guess=2000.0).T)

fig = go.Figure()
fig.add_scatter(x=phis2, y=Tref, mode="lines", name="standalone HP equilibrium",
                line=dict(color=COLORWAY[0], width=4), opacity=0.5)
fig.add_scatter(x=phis2, y=Tnet, mode="markers", name="network flame",
                marker=dict(color=COLORWAY[4], size=9))
fig.update_layout(
    title="Network flame vs. standalone equilibrium",
    xaxis_title="equivalence ratio  φ", yaxis_title="burnt T  [K]", height=440,
)
fig.show()

## Export for Nemo

The full network is available as `network` (a `Network`) and its converged mean flow as `sol` (a `Solution`).
Save either to a UI-readable YAML — `sol.to_yaml(path)` embeds the mean-flow result fields, `network.to_yaml(path)` writes the topology only — then open the file in Nemo.

In [ ]:
import os, tempfile

_out = os.path.join(tempfile.mkdtemp(), "reacting_flame.yaml")
sol.to_yaml(_out)  # embeds the mean-flow results; use network.to_yaml(_out) for topology only
print("saved case:", _out)